In [42]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace, HuggingFacePipeline
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [2]:
load_dotenv()

True

In [5]:
prompt1 = PromptTemplate(
    template='Generate a detailed report on {topic}',
    input_variables=['topic']
)

In [6]:
prompt2 = PromptTemplate(
    template='Generate a 5 pointer summary from the following text \n {text}',
    input_variables=['text']
)

In [9]:
parser = StrOutputParser()

In [30]:
repo_id = "deepseek-ai/DeepSeek-V4-Flash-0731"
llm = HuggingFaceEndpoint(
            repo_id=repo_id,
            task="text-generation", # Required for ChatHuggingFace mapping
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
        )
model = ChatHuggingFace(llm=llm)

In [31]:
chain = prompt1 | model | parser | prompt2 | model | parser

In [32]:
result = chain.invoke({'topic': 'reliability in research methods'})

In [33]:
print(result)

Here is a 5-pointer summary of the report:

- **Foundational Concept:** Reliability is a cornerstone of research methodology, referring to the consistency, stability, and reproducibility of measurement instruments and findings, and is essential for establishing research credibility and enabling replication.
- **Formal Definition:** In classical test theory, reliability is defined by the equation X = T + E (Observed score = True score + Error), with reliability quantified as the proportion of observed score variance attributable to true score variance.
- **Paradigm Distinction:** While traditionally associated with quantitative research, reliability holds equal importance in qualitative inquiry, albeit with different conceptualizations and assessment approaches.
- **Comprehensive Scope:** The report covers the concept's definitions, types, assessment methods, common threats, and strategies for enhancement across multiple research designs, including experimental, survey, observational, a

In [16]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
   +-----------------+     
   | ChatHuggingFace |     
   +-----------------+     
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

In [34]:
#parallel chains

In [51]:
prompt1 = PromptTemplate(
    template='Generate a 200 word essay on the topic: \n {topic}',
    input_variables=['topic']
)

In [52]:
prompt2 = PromptTemplate(
    template='Generate 5 Multiple Choice General Knowledge Questions regarding the topic: \n {topic}',
    input_variables=['topic']
)

In [53]:
prompt3 = PromptTemplate(
    template='Merge the provided essay and quiz into a single document \n essay -> {essay} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

In [54]:
parser = StrOutputParser()

In [55]:
parallel_chain = RunnableParallel({
    'essay': prompt1 | model | parser,
    'quiz': prompt2 | model | parser
})

In [56]:
merge_chain = prompt3 | model | parser

In [57]:
chain = parallel_chain | merge_chain

In [58]:
topic = 'Global Warming'

In [60]:
result = chain.invoke({'topic':topic})

In [61]:
print(result)

Here is the merged document, combining the essay and the quiz into a single, cohesive file.

---

### Global Warming: The Defining Challenge of Our Century

Global warming, driven by the relentless accumulation of greenhouse gases in our atmosphere, stands as the defining challenge of our century. Its consequences are no longer distant projections but observable realities: glaciers are receding at unprecedented rates, sea levels are rising, and weather patterns are becoming increasingly erratic, marked by more intense hurricanes, prolonged droughts, and devastating wildfires.

The primary culprit is the burning of fossil fuels for energy, alongside deforestation and industrial agriculture. This has created a delicate imbalance in the Earth's natural systems, trapping heat that would otherwise escape into space. The resulting warming disproportionately impacts the world's most vulnerable populations, threatening food and water security and displacing communities.

Addressing this crisis

In [50]:
chain.get_graph().print_ascii()

              +---------------------------+              
              | Parallel<notes,quiz>Input |              
              +---------------------------+              
                  ***               ***                  
               ***                     ***               
             **                           **             
+----------------+                    +----------------+ 
| PromptTemplate |                    | PromptTemplate | 
+----------------+                    +----------------+ 
          *                                   *          
          *                                   *          
          *                                   *          
+-----------------+                  +-----------------+ 
| ChatHuggingFace |                  | ChatHuggingFace | 
+-----------------+                  +-----------------+ 
          *                                   *          
          *                                   *          
          *   